In [ ]:
# ============================================================
# 环境配置
# - Colab 用户：取消注释下方 Colab 区块
# - 本地 Jupyter 用户：直接运行 Local 区块
# ============================================================

# ── Colab 环境（取消注释后运行） ──
# !pip install torch -q
# !pip install transformers -q
# !pip install matplotlib numpy -q

# ── 本地 Jupyter 环境 ──
import subprocess
import sys

def _install(pkg: str):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for package in ["torch", "transformers", "matplotlib", "numpy"]:
    _install(package)

In [ ]:
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

# GPT-2 代码实战：学习实现 vs 工程实现

基于论文 *Language Models are Unsupervised Multitask Learners*（Radford et al., 2019），
用 **零样本 Prompt 生成** 任务演示 GPT-2 的核心思想：**同一个 decoder-only 语言模型，仅通过改变输入提示就能尝试不同任务**。

本 Notebook 包含两条并行路径，并且使用 **同一组 Prompt** 做对照：

| | 学习路径 (Learning) | 工程路径 (Engineering) |
|---|---|---|
| 目标 | 理解 GPT-2 的关键模块与自回归生成逻辑 | 掌握工业级 GPT-2 的加载与推理方式 |
| 实现方式 | **L2：关键模块演示**（causal mask / Pre-LN / tiny decoder-only demo） | **E1：成熟库 + inference-only**（`transformers` + `openai-community/gpt2`） |
| 代码量 | 较多，显式展示 shape、mask、teacher forcing 与手写生成循环 | 较少，直接调用 `AutoTokenizer` 与 `model.generate()` |
| 适合场景 | 面试准备、原理拆解、理解训练/推理差异 | 工程验证、Prompt 实验、批量生成、快速原型 |

> 两条路径不是两套无关代码，而是同一套 GPT-2 思想在不同抽象层级上的表达。

## Section i：论文背景、任务定义与范围

### 1. 论文与任务
- **论文**：*Language Models are Unsupervised Multitask Learners*（OpenAI, 2019）
- **核心任务**：自回归语言建模，即根据前文预测下一个 token。
- **下游演示任务**：同一个模型面对不同 Prompt，分别尝试做文本续写、问答、摘要、翻译风格续写。

### 2. 为什么 GPT-2 是范式转折点
- GPT-1 更像“预训练后再微调”的代表。
- GPT-2 证明：当模型和数据规模足够大时，仅通过 Prompt 就能激活任务模式。
- 这把 NLP 从“每个任务配一个模型”推进到“一个模型面对多种任务”的方向。

### 3. 为什么这份 Notebook 选择 Prompt 生成作为主线
- 论文真正重要的不是“又一个 Transformer”，而是 **Prompt 范式 + zero-shot 使用方式**。
- 因此，这里不会把重点放在大规模预训练复现，而是放在：
  1. GPT-2 的关键结构为何适合稳定扩展；
  2. 训练时的 teacher forcing 与推理时的 autoregressive loop 如何对应；
  3. 工程上 `generate()` 到底封装了哪些步骤。

## Section ii：最小必要理论

### 1. 自回归语言建模目标

$$P(x_1, x_2, \dots, x_n) = \prod_{t=1}^{n} P(x_t \mid x_{<t})$$

直觉：模型不是一次性输出整句，而是**每一步只预测下一个 token**。

### 2. 因果自注意力

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V$$

其中 $M$ 是因果掩码（causal mask），用来保证位置 $t$ 只能看见自己和更早的位置。

### 3. GPT-2 的 Pre-LN Block

$$x_{out} = x + \text{Attn}(\text{LN}(x))$$
$$x_{out} = x_{mid} + \text{FFN}(\text{LN}(x_{mid}))$$

Pre-LN 的作用是让残差路径更“通”，训练深层模型时更稳定。

### 4. 训练 vs 推理
- **训练**：teacher forcing，给模型整段前缀并对所有位置同时算 loss。
- **推理**：每次只取最新 logits，采样 / 选最大值，然后把新 token 拼回输入继续生成。

> 这份 Notebook 只保留理解代码所需的最小理论；更完整的论文解读请回看配套 `04_GPT2：语言模型即无监督多任务学习器.md`。

### 组件映射表（Mandatory）

| 论文组件 | 学习路径覆盖 | 工程库 / API 对应 | 状态 |
|---|---|---|---|
| Token Embedding | `nn.Embedding` 显式实现 | `model.transformer.wte` | Runnable |
| Learned Positional Embedding | `nn.Embedding(max_len, d_model)` | `model.transformer.wpe` | Runnable |
| Scaled Dot-Product Causal Attention | 手写 mask + softmax 计算 | GPT-2 block 内部 attention | Runnable |
| Multi-Head Self-Attention | `CausalSelfAttention` 手写 | `model.transformer.h[i].attn` | Runnable |
| Pre-LN Decoder Block | `GPT2Block` 手写 | `model.transformer.h[i]` | Runnable |
| Final LayerNorm | `ln_f` 显式展示 | `model.transformer.ln_f` | Runnable |
| LM Head + Weight Tying | `lm_head.weight = tok_emb.weight` | `model.lm_head` | Runnable |
| Teacher Forcing | tiny batch loss demo | `model(..., labels=...)` / 训练流程 | Illustrative |
| Autoregressive Generation | 手写 `generate_tiny()` 循环 | `model.generate()` | Runnable |
| KV Cache / `past_key_values` | 只解释，不手写完整缓存逻辑 | `use_cache=True`, `past_key_values` | Explain-only |
| Prompt 作为任务接口 | 同一 tiny model 面对不同 prompt | 同一 pretrained GPT-2 面对不同 prompt | Runnable |

## Section 1：数据准备

学习路径不会尝试复现 WebText 规模训练，而是构造一个**极小的字符级语料**，其中同时包含：
- 普通文本续写样式
- `Q: ... A:` 问答样式
- `TL;DR:` 摘要样式
- `Translate English to French: ... => ...` 翻译样式

这样做的目标不是“让 tiny model 拥有真正的 zero-shot 能力”，而是让你在极小成本下看到：
1. teacher forcing 的训练数据长什么样；
2. 为什么 Prompt 会改变生成分布；
3. 同一套 Prompt 如何在工程路径里复用。

In [ ]:
SHARED_PROMPTS = [
    "The future of artificial intelligence is",
    "Knowledge is power. Q: What is power? A:",
    "Artificial intelligence helps people write code. TL;DR:",
    "Translate English to French: sea otter =>",
]

raw_text = """
The future of artificial intelligence is shaped by data and compute.
Prompting lets one model attempt many tasks.
Knowledge is power.
Artificial intelligence helps people write code and summarize documents.
The art of programming is the art of organizing complexity.
Translate English to French: sea otter => loutre de mer.
Translate English to French: cheese => fromage.
Knowledge is power. Q: What is power? A: power means the ability to act.
Artificial intelligence helps people write code. TL;DR: AI helps people write code.
Good models learn reusable patterns from text.
Questions and answers often appear together in natural documents.
Summaries often appear after markers such as TL;DR:.
""".strip()

raw_text = ((raw_text + "\n") * 40).strip()
vocab_source = raw_text + "\n" + "\n".join(SHARED_PROMPTS)

chars = sorted(set(vocab_source))
VOCAB_SIZE = len(chars)

char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}
UNKNOWN_CHAR = "?" if "?" in char_to_idx else chars[0]

def encode(text: str) -> list[int]:
    return [char_to_idx.get(ch, char_to_idx[UNKNOWN_CHAR]) for ch in text]

def decode(indices: list[int]) -> str:
    return "".join(idx_to_char[i] for i in indices)

print(f"Corpus length: {len(raw_text)} characters")
print(f"Vocabulary size: {VOCAB_SIZE}")
print("Shared prompt suite:")
for prompt in SHARED_PROMPTS:
    print("-", prompt)

In [ ]:
SEQ_LEN = 64
BATCH_SIZE = 32

class CharDataset(Dataset):
    """字符级语言建模数据集。
    input:  text[i : i + seq_len]
    target: text[i+1 : i + seq_len + 1]
    """
    def __init__(self, text: str, seq_len: int):
        self.data = torch.tensor(encode(text), dtype=torch.long)  # (total_chars,)
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.seq_len]                   # (seq_len,)
        y = self.data[idx + 1 : idx + self.seq_len + 1]           # (seq_len,)
        return x, y

dataset = CharDataset(raw_text, SEQ_LEN)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

sample_x, sample_y = next(iter(train_loader))
print(f"Input batch shape:  {tuple(sample_x.shape)}")
print(f"Target batch shape: {tuple(sample_y.shape)}")
print("\nSample input text:")
print(decode(sample_x[0].tolist()))
print("\nSample target text:")
print(decode(sample_y[0].tolist()))

In [ ]:
# ── Shared hyperparameters / 两条路径共用超参数 ──
D_MODEL = 128
NUM_HEADS = 4
NUM_LAYERS = 2
D_FF = 256
DROPOUT = 0.1
TINY_LR = 3e-4
TINY_TRAIN_STEPS = 40
MODEL_ID = "openai-community/gpt2"

print({
    "VOCAB_SIZE": VOCAB_SIZE,
    "SEQ_LEN": SEQ_LEN,
    "BATCH_SIZE": BATCH_SIZE,
    "D_MODEL": D_MODEL,
    "NUM_HEADS": NUM_HEADS,
    "NUM_LAYERS": NUM_LAYERS,
    "D_FF": D_FF,
    "TINY_LR": TINY_LR,
    "TINY_TRAIN_STEPS": TINY_TRAIN_STEPS,
    "MODEL_ID": MODEL_ID,
})

## Section 2：共享组件与学习路径入口

根据 skill 的可行性决策，这里选择：
- **学习路径深度**：L2（关键模块演示）
- **工程路径形式**：E1（成熟工业库）
- **工程动作**：inference-only

原因很直接：
- GPT-2 的真实价值来自大规模预训练后的 Prompt 能力，不适合在免费 Colab 上做“完整复现”；
- 但 GPT-2 的关键模块——因果注意力、Pre-LN、最终 LayerNorm、自回归生成——完全可以拆开并稳定演示；
- 工程上最真实、最稳定的入口就是直接使用 HuggingFace 的预训练 GPT-2 做推理。

## Section iii：学习路径（L2：关键模块演示）

### 组件 1：Token Embedding + Learned Positional Embedding

GPT-2 不是用固定正弦位置编码，而是使用**可学习的位置嵌入**：

$$h_0 = E_{token}(x) + E_{pos}(p)$$

输入 / 输出 shape：
- token ids：$(batch, seq)$
- token embedding：$(batch, seq, d_{model})$
- position embedding：$(1, seq, d_{model})$
- combined hidden states：$(batch, seq, d_{model})$

直觉：token embedding 负责“这个词是什么”，position embedding 负责“它在第几个位置”。

In [ ]:
class TinyEmbeddings(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, max_len: int, dropout: float):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device).unsqueeze(0)           # (1, T)
        tok = self.token_emb(idx)                                       # (B, T) -> (B, T, d_model)
        pos = self.pos_emb(pos)                                         # (1, T, d_model)
        return self.drop(tok + pos)                                     # (B, T, d_model)

embedding_layer = TinyEmbeddings(VOCAB_SIZE, D_MODEL, SEQ_LEN, DROPOUT).to(device)
embedding_out = embedding_layer(sample_x[:2].to(device))
print(f"Embedding output shape: {tuple(embedding_out.shape)}")

### 组件 2：Causal Self-Attention

GPT-2 的注意力不是普通 self-attention，而是 **causal self-attention**：当前位置只能看自己和左边。

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V$$

其中因果 mask $M$ 会把未来位置变成 $-\infty$。

输入 / 输出 shape：
- input hidden states：$(batch, seq, d_{model})$
- attention scores：$(batch, heads, seq, seq)$
- output hidden states：$(batch, seq, d_{model})$

这一步决定了 GPT-2 为什么能做严格的从左到右生成。

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float, max_len: int):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        mask = torch.triu(torch.ones(max_len, max_len), diagonal=1).bool()
        self.register_buffer("causal_mask", mask)

    def forward(self, x: torch.Tensor, return_attention: bool = False):
        B, T, C = x.shape                                               # (B, T, d_model)
        qkv = self.qkv_proj(x)                                          # (B, T, 3*d_model)
        q, k, v = qkv.chunk(3, dim=-1)                                  # each: (B, T, d_model)

        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2) # (B, H, T, Dh)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2) # (B, H, T, Dh)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2) # (B, H, T, Dh)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)   # (B, H, T, T)
        scores = scores.masked_fill(self.causal_mask[:T, :T], float("-inf"))
        attn = F.softmax(scores, dim=-1)                                 # (B, H, T, T)
        attn = self.attn_drop(attn)

        out = attn @ v                                                   # (B, H, T, Dh)
        out = out.transpose(1, 2).contiguous().view(B, T, C)             # (B, T, d_model)
        out = self.resid_drop(self.out_proj(out))                        # (B, T, d_model)
        if return_attention:
            return out, attn
        return out

attn_layer = CausalSelfAttention(D_MODEL, NUM_HEADS, DROPOUT, SEQ_LEN).to(device)
attn_layer.eval()
attn_out, attn_map = attn_layer(embedding_out, return_attention=True)
print(f"Attention output shape: {tuple(attn_out.shape)}")
print(f"Attention map shape:    {tuple(attn_map.shape)}")

### 组件 3：Pre-LN Decoder Block

GPT-2 和早期 Post-LN Transformer 的关键差异之一是 **LayerNorm 放在子层前面**：

$$x_{mid} = x + \text{Attn}(\text{LN}(x))$$
$$x_{out} = x_{mid} + \text{FFN}(\text{LN}(x_{mid}))$$

这让残差路径更利于梯度传播。对于深层 decoder-only 模型，这是很关键的稳定性改动。

In [ ]:
class GPT2Block(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float, max_len: int):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, num_heads, dropout, max_len)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))                                  # (B, T, d_model)
        x = x + self.ffn(self.ln2(x))                                   # (B, T, d_model)
        return x

block = GPT2Block(D_MODEL, NUM_HEADS, D_FF, DROPOUT, SEQ_LEN).to(device)
block_out = block(embedding_out)
print(f"Block output shape: {tuple(block_out.shape)}")

### 组件 4：Final LayerNorm + LM Head + Weight Tying

经过若干 decoder block 后，GPT-2 还会再做一次最终 LayerNorm：

$$h_f = \text{LN}(h_L)$$

然后通过语言模型头把 hidden states 投到词表维度：

$$\text{logits} = h_f W^T$$

其中一个常见工程技巧是 **weight tying**：输出层权重直接复用 token embedding 权重。这样能减少参数量，也让输入 / 输出词空间更一致。

In [ ]:
class TinyGPT2Demo(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, num_heads: int, num_layers: int, d_ff: int, max_len: int, dropout: float):
        super().__init__()
        self.max_len = max_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            GPT2Block(d_model, num_heads, d_ff, dropout, max_len)
            for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight                       # weight tying

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device).unsqueeze(0)           # (1, T)
        x = self.tok_emb(idx) + self.pos_emb(pos)                       # (B, T, d_model)
        x = self.drop(x)
        for block in self.blocks:
            x = block(x)                                                # (B, T, d_model)
        x = self.ln_f(x)                                                # (B, T, d_model)
        logits = self.lm_head(x)                                        # (B, T, vocab_size)
        return logits

tiny_model = TinyGPT2Demo(VOCAB_SIZE, D_MODEL, NUM_HEADS, NUM_LAYERS, D_FF, SEQ_LEN, DROPOUT).to(device)
tiny_logits = tiny_model(sample_x[:2].to(device))
print(f"Tiny GPT-2 logits shape: {tuple(tiny_logits.shape)}")

### 训练 vs 推理的区别（Learning Path）

GPT-2 最容易混淆的地方就在这里：

**训练阶段（teacher forcing）**
- 一次输入整段前缀
- 每个位置都预测“下一个 token”
- 所有位置可并行计算 loss

**推理阶段（autoregressive generation）**
- 只取最后一个位置的 logits
- 生成一个新 token 后，把它拼回输入
- 再进行下一步预测

下面先用 tiny model 做一个极小规模的 teacher-forcing 演示，再用手写循环做生成。

In [ ]:
optimizer = torch.optim.AdamW(tiny_model.parameters(), lr=TINY_LR)
tiny_loss_history = []

train_iter = iter(train_loader)
tiny_model.train()
for step in range(1, TINY_TRAIN_STEPS + 1):
    try:
        x_batch, y_batch = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        x_batch, y_batch = next(train_iter)

    x_batch = x_batch.to(device)                                         # (B, T)
    y_batch = y_batch.to(device)                                         # (B, T)

    logits = tiny_model(x_batch)                                         # (B, T, vocab_size)
    loss = F.cross_entropy(
        logits.reshape(-1, VOCAB_SIZE),                                  # (B*T, vocab_size)
        y_batch.reshape(-1)                                              # (B*T,)
    )

    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(tiny_model.parameters(), 1.0)
    optimizer.step()

    tiny_loss_history.append(loss.item())
    if step % 10 == 0 or step == 1:
        print(f"Step {step:02d}/{TINY_TRAIN_STEPS} - loss: {loss.item():.4f}")

plt.figure(figsize=(6, 4))
plt.plot(range(1, TINY_TRAIN_STEPS + 1), tiny_loss_history)
plt.title("Tiny GPT-2 Demo Loss")
plt.xlabel("Training step")
plt.ylabel("Loss")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Learning Path：手写自回归生成循环

下面的 `generate_tiny()` 就是 GPT-2 推理的最小形态：
1. 先编码 prompt；
2. 前向传播拿到最后一个位置的 logits；
3. 选下一个 token；
4. 拼回输入；
5. 重复这个过程。

这正是工程路径里 `model.generate()` 封装起来的核心循环。

In [ ]:
@torch.no_grad()
def generate_tiny(model: TinyGPT2Demo, prompt: str, max_new_tokens: int = 60, temperature: float = 0.8, top_k: int = 0) -> str:
    model.eval()
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)     # (1, T)

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -SEQ_LEN:]                                           # (1, <=SEQ_LEN)
        logits = model(idx_cond)                                                # (1, T, vocab_size)
        next_logits = logits[:, -1, :] / temperature                            # (1, vocab_size)

        if top_k > 0:
            values, _ = torch.topk(next_logits, min(top_k, next_logits.size(-1)))
            next_logits[next_logits < values[:, [-1]]] = float("-inf")

        probs = F.softmax(next_logits, dim=-1)                                  # (1, vocab_size)
        next_id = torch.multinomial(probs, num_samples=1)                       # (1, 1)
        idx = torch.cat([idx, next_id], dim=1)                                  # (1, T+1)

    return decode(idx[0].tolist())

print("Tiny model prompt demos")
print("-" * 80)
for prompt in SHARED_PROMPTS:
    tiny_output = generate_tiny(tiny_model, prompt, max_new_tokens=40, temperature=0.7, top_k=8)
    print(f"Prompt: {prompt}")
    print(f"Output: {tiny_output}")
    print("-" * 80)

## Section iv：工程路径（E1：成熟库 + inference-only）

### 工程决策
- **工程路径形式**：E1（成熟工业库）
- **工程动作**：inference-only
- **原因**：论文强调的是大规模预训练后的 zero-shot Prompt 能力，而不是小数据微调流程。对 GPT-2 来说，最真实的工程路径就是直接加载预训练模型并使用 `generate()`。

这里使用 HuggingFace 官方范式：
- `AutoTokenizer.from_pretrained()`
- `AutoModelForCausalLM.from_pretrained()`
- `model.generate()`

同时要注意 decoder-only 模型的一个工程细节：**批量生成时通常要左侧 padding，并把 `pad_token` 设成 `eos_token`**。

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

engineering_model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(device)
engineering_model.eval()

num_params = sum(p.numel() for p in engineering_model.parameters())
print(f"Loaded model: {MODEL_ID}")
print(f"Tokenizer vocab size: {tokenizer.vocab_size:,}")
print(f"Model parameters: {num_params:,}")
print(f"use_cache default: {engineering_model.config.use_cache}")

### 工程路径黑盒拆解（Black-box teardown）

#### `model.generate()` 实际在做什么？
1. tokenizer 把 prompt 变成 `input_ids` 和 `attention_mask`
2. 模型先对当前上下文做 forward
3. 取最后一个位置的 logits
4. 按 greedy / sampling / beam search 等策略选择下一个 token
5. 把新 token 拼回输入
6. 重复直到达到 `max_new_tokens` 或遇到停止条件

#### 它和学习路径如何对应？

| 学习路径实现 | 工程库内部对应 | 说明 |
|---|---|---|
| `TinyEmbeddings` | `model.transformer.wte/wpe` | token 与位置嵌入 |
| `CausalSelfAttention` | `model.transformer.h[i].attn` | decoder-only causal attention |
| `GPT2Block` | `model.transformer.h[i]` | Pre-LN block |
| `ln_f + lm_head` | `model.transformer.ln_f` + `model.lm_head` | 最终归一化与词表投影 |
| `generate_tiny()` | `model.generate()` | 手写版 = 最小工程版生成循环 |
| 解释型“未来 token 不可见” | `attention_mask` + causal logic | 库在内部自动处理 |
| 仅说明缓存 | `use_cache` / `past_key_values` | 避免每步重复算历史 KV |

In [ ]:
def generate_engineering(prompts, max_new_tokens: int = 40, do_sample: bool = True, temperature: float = 0.8, top_k: int = 50, num_beams: int = 1):
    model_inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128,
    ).to(device)

    generation_kwargs = dict(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        num_beams=num_beams,
        pad_token_id=tokenizer.eos_token_id,
        return_dict_in_generate=False,
    )
    if do_sample:
        generation_kwargs.update(temperature=temperature, top_k=top_k)

    with torch.no_grad():
        generated_ids = engineering_model.generate(**generation_kwargs)

    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

single_demo = generate_engineering([SHARED_PROMPTS[0]], max_new_tokens=40, do_sample=True, temperature=0.7, top_k=40)[0]
print("Single engineering demo")
print("Prompt:", SHARED_PROMPTS[0])
print("Output:", single_demo)

print("\nDecoding strategy comparison")
print("-" * 80)
strategy_outputs = {
    "greedy": generate_engineering([SHARED_PROMPTS[1]], max_new_tokens=30, do_sample=False, num_beams=1)[0],
    "sampling": generate_engineering([SHARED_PROMPTS[1]], max_new_tokens=30, do_sample=True, temperature=0.8, top_k=40, num_beams=1)[0],
    "beam_search": generate_engineering([SHARED_PROMPTS[1]], max_new_tokens=30, do_sample=False, num_beams=3)[0],
}
for name, text in strategy_outputs.items():
    print(f"[{name}] {text}")

### 批量推理、padding、缓存与参数权衡

对 decoder-only 模型，批量生成时要特别注意：
- **左侧 padding**：因为模型不是为了“从 padding token 之后继续写”而训练的；
- **`pad_token = eos_token`**：GPT-2 默认没有 pad token，工程上通常直接复用 EOS；
- **`attention_mask`**：让模型忽略 padding 部分；
- **`use_cache` / `past_key_values`**：推理时缓存历史 key/value，避免每一步重复计算整段上下文。

> 对开放式生成任务，`beam search` 可以演示搜索策略差异，但实际常见的创作式生成更常用 sampling。

| 因素 | 增大时 | 吞吐量 | 内存 | 速度 | 效果 |
|---|---|---|---|---|---|
| `batch_size` | 同时处理更多 prompt | ↑ | ↑↑ | ↑（单样本平均更快） | 通常不变 |
| `max_new_tokens` | 输出更长 | ↓ | ↑ | ↓ | 可能更完整 |
| `temperature` | 采样更随机 | ~ | ~ | ~ | 多样性↑、稳定性↓ |
| `top_k` | 候选集合更大 | ~ | ~ | ~ | 更开放，可能更发散 |
| `num_beams` | 搜索更宽 | ↓ | ↑ | ↓↓ | 某些任务更稳 |
| `use_cache=True` | 复用历史 KV | ↑ | ↑ | ↑↑ | 通常不变 |
| `device` CPU→GPU | 计算更并行 | ↑↑ | GPU-bound | ↑↑↑ | 通常不变 |

下面直接对同一批 prompts 做批量生成。

In [ ]:
batched_outputs = generate_engineering(SHARED_PROMPTS, max_new_tokens=36, do_sample=True, temperature=0.8, top_k=40)

print("Batched engineering outputs")
print("=" * 90)
for prompt, output in zip(SHARED_PROMPTS, batched_outputs):
    print(f"Prompt: {prompt}")
    print(f"Output: {output}")
    print("-" * 90)

## Section v：学习路径 vs 工程路径对比（Mandatory）

| 对比维度 | 学习路径 | 工程路径 |
|---|---|---|
| 教育价值 | 直接看到 causal mask、Pre-LN、teacher forcing、手写生成循环 | 直接理解工业级 API 怎样把这些步骤封装起来 |
| 代码量 | 较多，模块展开清楚 | 较少，几行代码即可运行预训练 GPT-2 |
| 灵活性 | 高：可以改 attention、block、采样逻辑、loss 计算 | 中：受限于库接口与预训练结构 |
| 稳定性 | 中：教学代码更容易受超参数与数据规模影响 | 高：大模型与 API 已经过大量验证 |
| 工业适配度 | 适合教学、面试、机制分析、小型原型 | 适合 Prompt 实验、批量生成、真实原型与生产前验证 |
| 适用场景 | 理解 decoder-only LLM 原理<br>解释训练 vs 推理差异<br>做局部机制实验 | 快速试 Prompt<br>比较解码策略<br>做批量推理和工程验证 |

## Section vi：训练 vs 推理差异（两条路径都要说清楚）

| 阶段 | 学习路径行为 | 工程路径行为 |
|---|---|---|
| 训练 | teacher forcing：输入整段前缀，目标序列右移一位，同时计算所有位置 loss | 本 Notebook 不训练预训练 GPT-2；真实训练由大规模离线预训练完成 |
| mask | 手写 causal mask，显式阻止当前位置看未来 token | 库在模型内部自动处理 causal attention |
| 推理 | `generate_tiny()` 每次只取最后一个位置 logits，然后把新 token 拼回输入 | `model.generate()` 封装了同样的自回归循环 |
| padding | tiny demo 只做单批字符序列，不强调 padding | 批量推理时需左侧 padding，并配合 `attention_mask` |
| cache | 这里只解释概念，不手写完整缓存逻辑 | `use_cache` / `past_key_values` 用于加速长序列生成 |
| 本质洞察 | 训练是并行算 loss，推理是串行生成 token | 同一算法，只是抽象层次更高、工程优化更多 |

> 关键洞察：`model.generate()` 不是“魔法函数”，只是把手写自回归循环、缓存、停止条件和解码策略一起封装了。

In [ ]:
print("Shared prompt comparison")
print("=" * 120)
for prompt in SHARED_PROMPTS:
    learning_text = generate_tiny(tiny_model, prompt, max_new_tokens=30, temperature=0.7, top_k=8)
    engineering_text = generate_engineering([prompt], max_new_tokens=30, do_sample=True, temperature=0.7, top_k=8)[0]

    print(f"Prompt:      {prompt}")
    print(f"Learning:    {learning_text[:180]}")
    print(f"Engineering: {engineering_text[:180]}")
    print("-" * 120)

# Appendix A: visualize causal attention structure
char_tokens = [idx_to_char[i] for i in sample_x[0][:16].tolist()]
heatmap = attn_map[0, 0, :16, :16].detach().cpu().numpy()

plt.figure(figsize=(6, 5))
plt.imshow(heatmap, cmap="magma")
plt.xticks(range(len(char_tokens)), char_tokens, rotation=45)
plt.yticks(range(len(char_tokens)), char_tokens)
plt.title("Causal Attention Pattern")
plt.xlabel("Key positions")
plt.ylabel("Query positions")
plt.colorbar()
plt.tight_layout()
plt.show()

## Section vii：结果、边界与适用条件

### 结果应该怎么读？
- **学习路径**的输出只说明：哪怕把模型缩到很小，GPT-2 的关键计算图——causal attention、Pre-LN、teacher forcing、自回归生成——仍然能跑通。
- **工程路径**的输出更接近论文真正强调的使用方式：固定参数，只靠 Prompt 改变任务行为。
- 同一组 Prompt 在两条路径下的表现差异，本质上反映的是**规模、数据和工程优化**，而不是“原理是否不同”。

### 条件、范围与限制
- **适用条件**：你关心的是 decoder-only 语言模型、Prompt 范式、训练/推理差异和工程生成流程。
- **范围边界**：这里不复现 WebText 训练，不代表 tiny demo 能展示 GPT-2 真正的 zero-shot 上限。
- **已知限制**：小字符级模型只能做机制演示；预训练 GPT-2 体量也远小于今天主流大模型，因此复杂推理和长上下文能力有限。

### 什么时候用哪条路径？
- 如果你在准备面试 / 课程讲解：先走**学习路径**。
- 如果你在做 Prompt 试验 / 批量生成 / 工程原型：直接走**工程路径**。
- 如果你在研究 LLM：先用学习路径理解生成循环和 mask，再去工程路径观察 cache、batching、解码策略的真实影响。

## Appendix A：可视化补充

上一个代码单元画出的热力图来自前面的**教学版 standalone causal attention 层**，用途是展示 mask 结构，而不是展示训练后语义注意力。

观察时重点看：
- 主对角线及其左下方区域有值；
- 右上方未来位置被 mask 掉，因此不会被关注；
- 这正是 decoder-only 语言模型能够稳定做自回归生成的结构前提。

## Section viii：面试与项目表达（Mandatory）

### 高频面试题

**Q1：GPT-2 相比 GPT-1 最大的突破是什么？**
- 不是单纯“参数更大”，而是证明了**大规模语言模型可以仅靠 Prompt 做零样本任务**。
- 这把范式从“预训练 + 微调”推进到“预训练 + Prompt”。

**Q2：为什么 GPT-2 必须使用 causal mask？**
- 因为语言模型训练目标是根据前文预测下一个 token。
- 如果当前位置能偷看未来 token，训练目标就被破坏了。

**Q3：Pre-LN 为什么比 Post-LN 更适合 GPT-2？**
- 深层网络中，Pre-LN 能让残差路径更顺畅地传梯度。
- 因此在很深的 decoder-only 模型里通常更稳定。

**Q4：teacher forcing 和 autoregressive generation 的本质区别是什么？**
- teacher forcing 在训练时并行计算所有位置 loss。
- autoregressive generation 在推理时串行地产生一个个 token。

**Q5：`model.generate()` 本质上在做什么？**
- 就是在循环执行：forward → 取最后一个 logits → 选择 token → 拼回输入。
- 只是库把缓存、停止条件、采样策略一起封装好了。

**Q6：为什么 GPT-2 批量生成时常把 `pad_token` 设成 `eos_token`？**
- 因为 GPT-2 默认没有 pad token。
- 工程上通常复用 EOS，并配合 `attention_mask` 忽略 padding。

**Q7：`use_cache` / `past_key_values` 的作用是什么？**
- 缓存历史 key/value，避免每生成一个 token 都重算整段历史注意力。
- 序列越长，加速效果越明显。

**Q8：为什么 beam search 不一定适合开放式文本生成？**
- beam 更偏向寻找高概率、稳定输出。
- 对创作式生成来说，sampling 往往更自然、更有多样性。

### 面试速答提纲

| # | 问题 | 一句话回答 |
|---|---|---|
| 1 | GPT-2 的核心贡献是什么？ | 证明了大规模语言模型仅靠 Prompt 就能做零样本多任务。 |
| 2 | causal mask 为什么必要？ | 防止当前位置偷看未来 token，保持自回归目标成立。 |
| 3 | Pre-LN 的价值是什么？ | 让深层残差路径更稳定，训练更容易收敛。 |
| 4 | `generate()` 本质在做什么？ | 封装好的自回归生成循环。 |
| 5 | `use_cache` 有什么用？ | 复用历史 KV，减少重复计算。 |
| 6 | 为什么要左侧 padding？ | decoder-only 模型不该从右侧 padding 后继续生成。 |
| 7 | GPT-2 和今天 LLM 的连续性是什么？ | 核心仍是 decoder-only + 自回归生成 + Prompt 接口。 |

### 项目表达

> 如果面试官问“你做过什么相关项目”，可以这样组织回答：

- **学习深度**：我从零拆过 GPT-2 的关键模块，包括 causal self-attention、Pre-LN block、最终 LayerNorm、weight tying 和自回归生成循环。
- **工程能力**：我用 HuggingFace 的预训练 GPT-2 做过 Prompt 实验，比较过 greedy、sampling、beam search 以及批量生成时的 padding / attention mask 处理。
- **对比思考**：我能说明手写学习版和工业版 `generate()` 的对应关系，也能解释为什么 cache、padding、解码策略会影响真实工程效果。

### 延伸阅读与对比

| 对比维度 | GPT-1 | GPT-2 | GPT-3 |
|---|---|---|---|
| 核心范式 | 预训练 + 微调 | Prompt 零样本 | Few-shot / in-context learning |
| 结构 | decoder-only Transformer | decoder-only Transformer | 更大规模 decoder-only Transformer |
| 关键亮点 | 证明预训练有效 | 证明 Prompt 可行 | 证明规模带来能力跃迁 |
| 典型使用 | 微调下游任务 | 零样本 Prompt | 少样本与更强泛化 |

### 进阶探索方向
- **KV cache 可视化**：继续拆解 `past_key_values` 在每层缓存了什么。
- **更细的解码策略比较**：系统比较 greedy、top-k、top-p、beam search 的差异。
- **从 GPT-2 到现代 LLM**：继续研究更长上下文、RoPE、Flash Attention、instruction tuning。
- **Prompt 工程实验**：系统测试不同 Prompt 模板如何改变输出分布。